# Notebook 02 — Territory Performance

**Project:** APAC Territory Planning  
**Objective:** Measure revenue and pipeline performance by territory — quota attainment, coverage ratio, win rate, and geographic distribution via choropleth maps.

**Business Question:** Are APAC territories balanced by opportunity?

In [1]:
import pandas as pd
import duckdb
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
import os

pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", "{:,.2f}".format)

DATA_DIR = "../data/raw"

In [2]:
accounts      = pd.read_csv(f"{DATA_DIR}/accounts.csv")
reps          = pd.read_csv(f"{DATA_DIR}/reps.csv")
assignments   = pd.read_csv(f"{DATA_DIR}/assignments.csv")
opportunities = pd.read_csv(f"{DATA_DIR}/opportunities.csv")
customers     = pd.read_csv(f"{DATA_DIR}/customers.csv")

customers = customers[
    (pd.to_datetime(customers["contract_end"]) > pd.Timestamp("2025-03-01")) |
    (customers["renewal_flag"] == 1)
].copy()

print(f"Active customers (filtered): {len(customers):,}")
print(f"Dropped: {len(pd.read_csv(f'{DATA_DIR}/customers.csv')) - len(customers):,} churned/expired")

con = duckdb.connect()
con.register("accounts",      accounts)
con.register("reps",          reps)
con.register("assignments",   assignments)
con.register("opportunities", opportunities)
con.register("customers",     customers)

print("Tables registered:")
con.execute("SHOW TABLES").df()

Active customers (filtered): 281
Dropped: 131 churned/expired
Tables registered:


,name
0,accounts
1,assignments
2,customers
3,opportunities
4,reps


## 1. Quota Attainment by Rep

In [3]:
sql = """
SELECT
    r.rep_id,
    r.rep_name,
    r.subregion,
    r.segment_focus,
    r.quota_usd,
    COALESCE(SUM(c.arr), 0)                                AS actual_arr,
    COALESCE(SUM(c.arr), 0) / r.quota_usd * 100            AS quota_attainment_pct,
    COUNT(DISTINCT c.customer_id)                          AS n_customers
FROM reps r
LEFT JOIN customers c ON r.rep_id = c.rep_id
WHERE r.subregion != 'Regional'
GROUP BY r.rep_id, r.rep_name, r.subregion, r.segment_focus, r.quota_usd
ORDER BY quota_attainment_pct DESC
"""

os.makedirs("../sql", exist_ok=True)
with open("../sql/quota_attainment.sql", "w") as f:
    f.write(sql)

quota = con.execute(sql).df()

print("QUOTA ATTAINMENT BY REP")
print(f"{'Rep':<22} {'Subregion':<16} {'Segment':<12} {'Quota':>10} {'Actual ARR':>11} {'Attain%':>8} {'Custs':>6}")
print("─" * 90)
for _, row in quota.iterrows():
    print(f"{row['rep_name']:<22} {row['subregion']:<16} {row['segment_focus']:<12} "
          f"{row['quota_usd']:>10,.0f} {row['actual_arr']:>11,.0f} "
          f"{row['quota_attainment_pct']:>7.1f}% {row['n_customers']:>6,.0f}")

QUOTA ATTAINMENT BY REP
Rep                    Subregion        Segment           Quota  Actual ARR  Attain%  Custs
──────────────────────────────────────────────────────────────────────────────────────────
Sara Ross              SEA              SMB             400,000   1,177,100   294.3%     20
Jacob Wilson           ANZ              Mid-Market      700,000   1,480,600   211.5%     20
Michael Roberts        SEA              SMB             400,000     572,100   143.0%     20
Arthur Miller          Greater China    SMB             400,000     517,900   129.5%     14
Anthony Mccarty        Greater China    SMB             400,000     487,300   121.8%      7
Lauren Butler          SEA              SMB             400,000     422,600   105.7%     18
Timothy Lawson         North Asia       Mid-Market      903,000     664,800    73.6%     15
Teresa Robbins         ANZ              Enterprise    1,200,000     414,200    34.5%      5
Bill Sullivan DVM      Greater China    Enterprise    1,2

## 2. Pipeline Coverage Ratio by Subregion

In [4]:
sql = """
WITH rep_quotas AS (
    SELECT
        subregion,
        SUM(quota_usd) AS total_quota
    FROM reps
    WHERE subregion != 'Regional'
    GROUP BY subregion
),
opp_pipeline AS (
    SELECT
        a.subregion,
        COALESCE(SUM(o.arr_potential)
            FILTER (WHERE o.stage NOT IN ('Closed Won','Closed Lost')), 0) AS open_pipeline,
        COUNT(DISTINCT o.opportunity_id)
            FILTER (WHERE o.stage NOT IN ('Closed Won','Closed Lost'))      AS open_opps,
        COUNT(DISTINCT o.opportunity_id)
            FILTER (WHERE o.stage = 'Closed Won')                           AS closed_won,
        COUNT(DISTINCT o.opportunity_id)
            FILTER (WHERE o.stage = 'Closed Lost')                          AS closed_lost
    FROM opportunities o
    JOIN accounts a ON o.account_id = a.account_id
    GROUP BY a.subregion
)
SELECT
    r.subregion,
    r.total_quota,
    p.open_pipeline,
    p.open_pipeline / r.total_quota * 100                  AS coverage_ratio,
    p.open_opps,
    p.closed_won,
    p.closed_lost,
    ROUND(p.closed_won * 100.0 / NULLIF(p.closed_won + p.closed_lost, 0), 1) AS win_rate_pct
FROM rep_quotas r
JOIN opp_pipeline p ON r.subregion = p.subregion
ORDER BY coverage_ratio DESC
"""

with open("../sql/pipeline_coverage.sql", "w") as f:
    f.write(sql)

pipeline = con.execute(sql).df()

print("PIPELINE COVERAGE BY SUBREGION")
print(f"{'Subregion':<18} {'Total Quota':>12} {'Open Pipeline':>14} {'Coverage%':>10} {'Win Rate':>9}")
print("─" * 68)
for _, row in pipeline.iterrows():
    print(f"{row['subregion']:<18} {row['total_quota']:>12,.0f} {row['open_pipeline']:>14,.0f} "
          f"{row['coverage_ratio']:>9.1f}% {row['win_rate_pct']:>8.1f}%")

PIPELINE COVERAGE BY SUBREGION
Subregion           Total Quota  Open Pipeline  Coverage%  Win Rate
────────────────────────────────────────────────────────────────────
Greater China         2,400,000     11,681,300     486.7%     32.7%
SEA                   4,300,000     14,100,200     327.9%     23.1%
India                 1,200,000      1,999,200     166.6%     36.4%
ANZ                   1,900,000      2,496,200     131.4%      0.0%
North Asia            2,591,000      2,903,000     112.0%     22.2%


## 3. Revenue by Territory — Choropleth Map

In [5]:
sql = """
SELECT
    a.country,
    a.subregion,
    COUNT(DISTINCT c.customer_id)   AS n_customers,
    COALESCE(SUM(c.arr), 0)         AS total_arr,
    COALESCE(AVG(c.arr), 0)         AS avg_arr
FROM accounts a
LEFT JOIN customers c ON a.account_id = c.account_id
GROUP BY a.country, a.subregion
ORDER BY total_arr DESC
"""

with open("../sql/arr_by_country.sql", "w") as f:
    f.write(sql)

arr_country = con.execute(sql).df()

# ISO alpha-3 mapping for choropleth
iso_map = {
    "SG": "SGP", "ID": "IDN", "MY": "MYS", "TH": "THA", "PH": "PHL", "VN": "VNM",
    "CN": "CHN", "HK": "HKG", "TW": "TWN",
    "JP": "JPN", "KR": "KOR",
    "IN": "IND",
    "AU": "AUS", "NZ": "NZL"
}

arr_country["iso_alpha"] = arr_country["country"].map(iso_map)

fig = px.choropleth(
    arr_country,
    locations="iso_alpha",
    color="total_arr",
    hover_name="country",
    hover_data={"n_customers": True, "total_arr": ":,.0f", "avg_arr": ":,.0f", "iso_alpha": False},
    color_continuous_scale="Blues",
    title="Customer ARR by Country",
    scope="asia"
)

fig.update_layout(
    height=550,
    font=dict(size=13),
    coloraxis_colorbar=dict(title="Total ARR (USD)")
)

fig.show()

os.makedirs("../outputs", exist_ok=True)
fig.write_image("../outputs/02_arr_by_country.png", width=900, height=550, scale=2)
print("Saved: ../outputs/02_arr_by_country.png")

Saved: ../outputs/02_arr_by_country.png


In [6]:
print("CUSTOMER ARR BY COUNTRY")
print(f"{'Country':<10} {'Subregion':<18} {'Customers':>10} {'Total ARR':>12} {'Avg ARR':>10}")
print("─" * 55)
for _, row in arr_country.sort_values("total_arr", ascending=False).iterrows():
    print(f"{row['country']:<10} {row['subregion']:<18} {row['n_customers']:>10,.0f} "
          f"{row['total_arr']:>12,.0f} {row['avg_arr']:>10,.0f}")

CUSTOMER ARR BY COUNTRY
Country    Subregion           Customers    Total ARR    Avg ARR
───────────────────────────────────────────────────────
NZ         ANZ                        26    1,656,800     63,723
JP         North Asia                 41    1,348,200     32,883
KR         North Asia                 41    1,242,600     30,307
AU         ANZ                        18    1,085,800     60,322
TW         Greater China              16      866,700     54,169
MY         SEA                        16      777,700     48,606
SG         SEA                        14      555,500     39,679
TH         SEA                        14      483,800     34,557
ID         SEA                        16      461,000     28,812
CN         Greater China              14      448,600     32,043
VN         SEA                        18      427,900     23,772
HK         Greater China               8      258,400     32,300
PH         SEA                        17      180,800     10,635
IN        

## 4. Open Pipeline by Country

In [7]:
sql = """
SELECT
    a.country,
    a.subregion,
    COUNT(DISTINCT o.opportunity_id)    AS n_opps,
    COALESCE(SUM(o.arr_potential), 0)   AS open_pipeline
FROM accounts a
LEFT JOIN opportunities o ON a.account_id = o.account_id
    AND o.stage NOT IN ('Closed Won', 'Closed Lost')
GROUP BY a.country, a.subregion
ORDER BY open_pipeline DESC
"""

with open("../sql/pipeline_by_country.sql", "w") as f:
    f.write(sql)

pipeline_country = con.execute(sql).df()
pipeline_country["iso_alpha"] = pipeline_country["country"].map(iso_map)

fig2 = px.choropleth(
    pipeline_country,
    locations="iso_alpha",
    color="open_pipeline",
    hover_name="country",
    hover_data={"n_opps": True, "open_pipeline": ":,.0f", "iso_alpha": False},
    color_continuous_scale="Oranges",
    title="Open Pipeline by Country",
    scope="asia"
)

fig2.update_layout(
    height=550,
    font=dict(size=13),
    coloraxis_colorbar=dict(title="Open Pipeline (USD)")
)

fig2.show()

fig2.write_image("../outputs/02_pipeline_by_country.png", width=900, height=550, scale=2)
print("Saved: ../outputs/02_pipeline_by_country.png")

Saved: ../outputs/02_pipeline_by_country.png


## 5. Quota Attainment by Subregion — Bar Chart

In [8]:
subregion_perf = (
    quota.groupby("subregion")
    .agg(
        total_quota=("quota_usd", "sum"),
        total_arr=("actual_arr", "sum"),
        n_reps=("rep_id", "count")
    )
    .assign(attainment_pct=lambda x: x["total_arr"] / x["total_quota"] * 100)
    .sort_values("attainment_pct", ascending=False)
    .reset_index()
)

max_val = subregion_perf["attainment_pct"].max()

fig3 = go.Figure()

fig3.add_trace(go.Bar(
    x=subregion_perf["subregion"],
    y=subregion_perf["attainment_pct"],
    text=[f"{v:.0f}%" for v in subregion_perf["attainment_pct"]],
    textposition="outside",
    marker_color=["#2196F3" if v >= 100 else "#FF5722" for v in subregion_perf["attainment_pct"]]
))

fig3.update_layout(
    title="Quota Attainment by Subregion",
    xaxis_title="Subregion",
    yaxis_title="Attainment %",
    yaxis=dict(range=[0, max_val * 1.25]),
    height=500,
    font=dict(size=13),
    showlegend=False
)

fig3.add_hline(
    y=100,
    line_dash="dash",
    line_color="grey",
    annotation_text="100% quota",
    annotation_position="top right"
)

fig3.show()

fig3.write_image("../outputs/02_quota_attainment.png", width=900, height=500, scale=2)
print("Saved: ../outputs/02_quota_attainment.png")

Saved: ../outputs/02_quota_attainment.png


## Key Findings

1. **Most reps below 100% quota attainment:** Filtering to active customers only reveals true performance — only SEA SMB reps exceeding quota (105-294%), confirming Enterprise and Mid-Market reps are not converting enough customers
2. **India reps at the bottom (6-16% attainment):** Lowest in APAC — under-penetration confirmed, SMB market not converting to active customers
3. **Enterprise reps all below 35% attainment:** Complex deals, longer cycles, fewer active customers — Enterprise territories need higher-value account enrichment not more accounts
4. **Greater China leads pipeline coverage (487%):** Strong open pipeline but win rate only 32.7% — conversion execution is the gap
5. **North Asia pipeline weakest (112%):** Barely above 1x quota — pipeline generation needs attention despite reasonable customer base in JP and KR
6. **ANZ win rate 0%:** Insufficient closed opportunity sample to calculate — not a reliable signal
7. **Active customer filter applied:** 131 churned/expired customers excluded — attainment reflects current revenue only, not historical cumulative ARR